# ShopAssist AI 2.0 — Function Calling Edition

This notebook upgrades the original ShopAssist AI to **ShopAssist 2.0** by integrating OpenAI's **Function Calling API**.

## What Changed vs v1?

| Layer | v1 Approach | v2 Approach |
|---|---|---|
| User Requirements Extraction | Multi-step: `intent_confirmation_layer()` → `dictionary_present()` | Single function call: `extract_user_requirements` tool |
| Laptop Feature Classification | `product_map_layer()` with JSON prompt hacks | Clean function call: `classify_laptop_features` tool |
| Conversation Control | `intent_confirmation_layer()` as a separate LLM call | Built-in tool invocation as the signal |
| Layers Removed | N/A | `intent_confirmation_layer()`, `dictionary_present()` (merged) |

**Benefits:** Fewer LLM calls, strongly-typed structured output, more reliable parsing, no JSON prompt hacks.

## Part 1: Setup & Imports

In [1]:
!pip install -U -q openai tenacity pandas

zsh:1: command not found: pip


In [2]:
import os
import json
import pandas as pd
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt

# ── Load API key ──────────────────────────────────────────────────────────────
openai.api_key = open("openai_key.txt", "r").read().strip()
os.environ['OPENAI_API_KEY'] = openai.api_key

MODEL = 'gpt-3.5-turbo'   # swap to gpt-4o for better reliability
print("Setup complete.")

Setup complete.


## Part 2: Function (Tool) Definitions

OpenAI Function Calling lets us describe a Python function as a JSON schema.  
The model decides *when* to call the function and fills the arguments — no prompt
gymnastics needed to get a dictionary out of a string.

We define **two tools**:
1. `submit_user_requirements` — called by the model when it has gathered all 6 user attributes.
2. `classify_laptop_features` — called to extract structured specs from a laptop description.

In [3]:
# ── Tool 1: User Requirements Extraction ─────────────────────────────────────
# The model calls this tool ONLY when it is confident all 6 values are captured.
# Replaces: intent_confirmation_layer() + dictionary_present()

USER_REQUIREMENTS_TOOL = {
    "type": "function",
    "function": {
        "name": "submit_user_requirements",
        "description": (
            "Call this function ONLY when you have confidently gathered ALL six "
            "user requirements through the conversation. Do NOT call it until the "
            "budget and all five feature ratings are confirmed."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "GPU intensity": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Importance of graphics processing power for the user."
                },
                "Display quality": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Importance of display resolution and colour accuracy."
                },
                "Portability": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How lightweight/portable the laptop needs to be."
                },
                "Multitasking": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Importance of running many applications simultaneously."
                },
                "Processing speed": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Importance of CPU performance for compute-heavy tasks."
                },
                "Budget": {
                    "type": "integer",
                    "description": "User's maximum budget in INR (must be >= 25000)."
                }
            },
            "required": [
                "GPU intensity", "Display quality", "Portability",
                "Multitasking", "Processing speed", "Budget"
            ]
        }
    }
}


# ── Tool 2: Laptop Feature Classification ────────────────────────────────────
# Replaces: product_map_layer()

LAPTOP_CLASSIFIER_TOOL = {
    "type": "function",
    "function": {
        "name": "classify_laptop_features",
        "description": (
            "Extract and classify laptop features from the provided description "
            "into five standardised ratings."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "GPU intensity": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": (
                        "low = integrated / Intel UHD; "
                        "medium = M1/AMD Radeon/Intel Iris; "
                        "high = Nvidia RTX or similar."
                    )
                },
                "Display quality": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": (
                        "low = below 1920x1080; "
                        "medium = 1920x1080 (Full HD); "
                        "high = 4K / Retina / HDR."
                    )
                },
                "Portability": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": (
                        "high = < 1.51 kg; "
                        "medium = 1.51–2.51 kg; "
                        "low = > 2.51 kg."
                    )
                },
                "Multitasking": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": (
                        "low = 8–12 GB RAM; "
                        "medium = 16 GB RAM; "
                        "high = 32+ GB RAM."
                    )
                },
                "Processing speed": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": (
                        "low = Core i3 / Ryzen 3; "
                        "medium = Core i5 / Ryzen 5; "
                        "high = Core i7/i9 / Ryzen 7/9."
                    )
                }
            },
            "required": [
                "GPU intensity", "Display quality", "Portability",
                "Multitasking", "Processing speed"
            ]
        }
    }
}

print("Tool definitions ready.")

Tool definitions ready.


## Part 3: Core Helper Functions

In [4]:
# ── Moderation check (unchanged from v1) ──────────────────────────────────────
def moderation_check(text: str) -> str:
    """Returns 'Flagged' or 'Not Flagged'."""
    result = openai.moderations.create(input=text)
    return "Flagged" if result.results[0].flagged else "Not Flagged"


# ── Generic chat completions wrapper (used for Stage 3 only now) ───────────────
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_chat_completions(messages: list, tools: list = None) -> str:
    """
    Thin wrapper around the Chat Completions API.
    Returns the assistant's text reply (tool calls handled separately).
    """
    kwargs = dict(model=MODEL, messages=messages, seed=2345)
    if tools:
        kwargs["tools"] = tools
    response = openai.chat.completions.create(**kwargs)
    return response.choices[0].message.content or ""


print("Helper functions ready.")

Helper functions ready.


## Part 4: Stage 1 — Intent Clarity via Function Calling

### How it works
- The system prompt tells the assistant to keep asking questions until it can call `submit_user_requirements`.
- When the model is ready, it emits a `tool_calls` response (no text). We parse the arguments directly — no extra LLM call needed to validate or extract a dictionary.
- **Layers removed:** `intent_confirmation_layer()` and `dictionary_present()` are gone.

In [5]:
def initialize_conversation() -> list:
    """
    Build the initial system message for Stage 1.
    The assistant converses until it invokes `submit_user_requirements`.
    """
    system_message = """
You are an intelligent laptop shopping assistant. Your goal is to recommend the
perfect laptop by first understanding what the customer needs.

You must gather values for SIX attributes through natural conversation:
  1. GPU intensity      (low / medium / high)
  2. Display quality    (low / medium / high)
  3. Portability        (low / medium / high)
  4. Multitasking       (low / medium / high)
  5. Processing speed   (low / medium / high)
  6. Budget             (a number in INR, minimum 25,000)

Guidance:
- Ask one or two questions at a time; keep the conversation natural and concise.
- Infer values from context where possible (e.g. "I play AAA games" → high GPU).
- If the user states a budget below 25,000 INR, inform them that no laptops are
  available in that range and ask for a revised budget.
- Once you are CONFIDENT about all six values, call the `submit_user_requirements`
  function. Do NOT call it until you have confirmed every attribute.

Start the conversation with a friendly welcome and ask the user what they plan
to use the laptop for.
"""
    return [{"role": "system", "content": system_message}]


@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def chat_with_function_calling(messages: list):
    """
    Send the conversation to the API with the user-requirements tool attached.

    Returns:
        (reply_text, tool_args_or_None)
        - reply_text  : assistant's conversational message (may be empty string)
        - tool_args   : dict of extracted requirements if tool was called, else None
    """
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[USER_REQUIREMENTS_TOOL],
        # 'auto' lets the model decide; use 'required' to force extraction at end
        tool_choice="auto",
        seed=1234
    )

    choice = response.choices[0]
    assistant_msg = choice.message

    # Append the assistant turn (needed for multi-turn history)
    messages.append(assistant_msg.to_dict() if hasattr(assistant_msg, 'to_dict')
                    else {"role": "assistant",
                          "content": assistant_msg.content or "",
                          "tool_calls": [tc.to_dict() for tc in (assistant_msg.tool_calls or [])]})

    # Did the model call our tool?
    if choice.finish_reason == "tool_calls" and assistant_msg.tool_calls:
        tc = assistant_msg.tool_calls[0]  # we only have one tool
        tool_args = json.loads(tc.function.arguments)
        # Ensure Budget is an int
        tool_args['Budget'] = int(str(tool_args['Budget']).replace(',', '').split()[0])
        return "", tool_args

    # Otherwise return the conversational text
    return assistant_msg.content or "", None


print("Stage 1 functions ready.")

Stage 1 functions ready.


### Stage 1 — Quick Test

In [6]:
# Demo: run a short scripted conversation to show the function call firing
test_conv = initialize_conversation()

# Assistant welcome
welcome, _ = chat_with_function_calling(test_conv)
print("Assistant:", welcome)

scripted_inputs = [
    "I do 3D rendering and video editing professionally.",
    "I work from my studio, so I don't need to carry it around.",
    "I need the highest display quality available.",
    "I run many heavy apps simultaneously, so max RAM is important.",
    "My budget is around 1,50,000 INR."
]

user_requirements = None
for user_text in scripted_inputs:
    print(f"\nUser: {user_text}")
    test_conv.append({"role": "user", "content": user_text})
    reply, tool_args = chat_with_function_calling(test_conv)
    if tool_args:
        user_requirements = tool_args
        print("\n✅ Function called! Extracted requirements:")
        print(json.dumps(user_requirements, indent=2))
        break
    else:
        print(f"Assistant: {reply}")

Assistant: Hello! I'm here to help you find the perfect laptop. What do you plan to use the laptop for?

User: I do 3D rendering and video editing professionally.
Assistant: Great! To help you find the best laptop for your needs, could you please tell me how important GPU intensity is for you? Would you say it's low, medium, or high on your priority list?

User: I work from my studio, so I don't need to carry it around.
Assistant: That's good to know! Since portability is not a top concern for you, how would you rate the display quality you need for your 3D rendering and video editing tasks? Is it low, medium, or high on your priority list?

User: I need the highest display quality available.
Assistant: Got it, high display quality is crucial for your work. How about multitasking? Do you often run multiple applications simultaneously while working, or do you prefer to focus on one task at a time?

User: I run many heavy apps simultaneously, so max RAM is important.
Assistant: Understoo

## Part 5: Stage 2 — Product Mapping via Function Calling

`classify_laptop_features` replaces `product_map_layer()`.  
The schema-enforced enum values eliminate the need for regex/JSON hacks.

In [7]:
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def classify_laptop_features(description: str) -> dict:
    """
    Classify a laptop description into 5 feature ratings using Function Calling.
    Replaces product_map_layer() from v1.

    Returns a dict with keys:
      GPU intensity, Display quality, Portability, Multitasking, Processing speed
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a laptop specifications classifier. "
                "Read the description and call `classify_laptop_features` with the correct ratings. "
                "You MUST call the function — do not reply in text."
            )
        },
        {
            "role": "user",
            "content": f"Classify this laptop:\n{description}"
        }
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[LAPTOP_CLASSIFIER_TOOL],
        tool_choice={"type": "function", "function": {"name": "classify_laptop_features"}},
        seed=1234
    )

    tool_call = response.choices[0].message.tool_calls[0]
    return json.loads(tool_call.function.arguments)


def build_laptop_catalogue(csv_path: str = 'laptop_data.csv') -> pd.DataFrame:
    """
    Load laptop data and enrich with structured feature columns.
    Results are cached to 'updated_laptop_v2.csv'.
    """
    cache = 'updated_laptop_v2.csv'
    if os.path.exists(cache):
        print(f"Loading cached catalogue from {cache}")
        return pd.read_csv(cache)

    df = pd.read_csv(csv_path)
    print(f"Classifying {len(df)} laptops...")
    df['laptop_feature'] = df['Description'].apply(classify_laptop_features)
    df.to_csv(cache, index=False)
    print("Done. Saved to", cache)
    return df


# ── Quick classification test ─────────────────────────────────────────────────
sample_desc = (
    "The Dell Inspiron features an Intel Core i5 processor at 2.4 GHz, 8 GB RAM, "
    "SSD, 15.6\" 1920x1080 LCD, Intel UHD GPU, and weighs 2.5 kg. Price: 35,000 INR."
)
print("Classification result:")
print(json.dumps(classify_laptop_features(sample_desc), indent=2))

Classification result:
{
  "GPU intensity": "low",
  "Display quality": "medium",
  "Portability": "low",
  "Multitasking": "low",
  "Processing speed": "medium"
}


In [8]:
# ── Scoring & top-3 selection (same logic as v1, but cleaner inputs) ──────────

SCORE_MAP = {'low': 0, 'medium': 1, 'high': 2}


def compare_laptops_with_user(user_req: dict, df: pd.DataFrame) -> str:
    """
    Score each laptop against user requirements and return the top-3 as JSON.

    v2 change: accepts the already-structured user_req dict directly — no
    extra dictionary_present() call needed because Function Calling already
    delivered a clean dict.
    """
    budget = int(str(user_req.get('Budget', 0)).replace(',', '').split()[0])

    # Filter by budget
    filtered = df.copy()
    filtered['Price'] = (
        filtered['Price'].astype(str)
        .str.replace(',', '')
        .str.split().str[0]
        .astype(int)
    )
    filtered = filtered[filtered['Price'] <= budget].copy()

    if filtered.empty:
        return json.dumps([])

    filtered['Score'] = 0

    for idx, row in filtered.iterrows():
        # laptop_feature is already a dict (stored as string in CSV, parse it)
        raw = row['laptop_feature']
        if isinstance(raw, str):
            try:
                laptop_spec = json.loads(raw.replace("'", '"'))
            except Exception:
                laptop_spec = {}
        else:
            laptop_spec = raw

        score = 0
        feature_keys = ['GPU intensity', 'Display quality', 'Portability',
                        'Multitasking', 'Processing speed']
        for key in feature_keys:
            laptop_val = SCORE_MAP.get(laptop_spec.get(key, 'low'), 0)
            user_val   = SCORE_MAP.get(user_req.get(key, 'low'), 0)
            if laptop_val >= user_val:
                score += 1

        filtered.at[idx, 'Score'] = score

    top3 = (
        filtered
        .drop('laptop_feature', axis=1)
        .sort_values('Score', ascending=False)
        .head(3)
    )
    return top3.to_json(orient='records')


def recommendation_validation(top3_json: str) -> list:
    """Keep only laptops with Score > 2."""
    data = json.loads(top3_json)
    return [item for item in data if item.get('Score', 0) > 2]


print("Stage 2 functions ready.")

Stage 2 functions ready.


## Part 6: Stage 3 — Product Recommendation

In [9]:
def initialize_conv_reco(products: list) -> list:
    """
    Start the recommendation conversation with the validated laptop list.
    (Unchanged from v1 — this layer didn't need Function Calling.)
    """
    system_message = """
You are an intelligent laptop gadget expert helping a user choose from a
shortlisted set of laptops. Answer questions helpfully and honestly.

Present the laptops in this format (descending price):
  1. <Name>: <key specs>, Price: Rs <price>
  2. ...

Keep answers concise. If no laptops matched the user's requirements, suggest
they speak to a human expert.
"""
    user_message = f"Here are the recommended laptops: {json.dumps(products, indent=2)}"
    return [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message}
    ]


print("Stage 3 functions ready.")

Stage 3 functions ready.


## Part 7: Dialogue Management System (ShopAssist 2.0)

The main loop is simpler than v1:
- No manual `intent_confirmation_layer()` check — the function call IS the confirmation.
- No `dictionary_present()` call — the function arguments ARE the dictionary.
- Fewer total LLM API calls per session.

In [14]:
def dialogue_mgmt_system_v2():
    """
    ShopAssist 2.0 — main dialogue loop using Function Calling.

    Key improvements over v1:
    - Removed intent_confirmation_layer() — tool call IS the confirmation.
    - Removed dictionary_present()        — tool args ARE the parsed dict.
    - Removed separate JSON-mode calls    — schema enforcement built into tool.
    """
    # ── Pre-load laptop catalogue once ──────────────────────────────────────
    laptop_df = build_laptop_catalogue()

    # ── Stage 1: Initialise conversation ────────────────────────────────────
    conversation = initialize_conversation()

    # Get opening welcome message
    welcome, _ = chat_with_function_calling(conversation)
    print("\nAssistant:", welcome, "\n")

    user_requirements = None
    conversation_reco = None

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == "exit":
            print("\nAssistant: Thank you for using ShopAssist! Goodbye.")
            break

        # Moderation
        if moderation_check(user_input) == 'Flagged':
            print("Assistant: Sorry, that message was flagged. Please rephrase.\n")
            continue

        # ── Stage 1: Still gathering requirements ────────────────────────
        if user_requirements is None:
            conversation.append({"role": "user", "content": user_input})
            reply, tool_args = chat_with_function_calling(conversation)

            if moderation_check(reply) == 'Flagged':
                print("Assistant: Sorry, an issue occurred. Please restart.\n")
                break

            if tool_args:
                # ✅ Function called — requirements captured
                user_requirements = tool_args
                print(f"\n✅ Requirements captured: {user_requirements}\n")
                print("Assistant: Thank you! Let me fetch the best laptops for you...\n")

                # ── Stage 2: Product matching ────────────────────────────
                top3_json = compare_laptops_with_user(user_requirements, laptop_df)
                validated  = recommendation_validation(top3_json)

                if not validated:
                    print(
                        "Assistant: I couldn't find laptops matching all your criteria. "
                        "I'd recommend speaking to one of our human experts for a more "
                        "tailored suggestion.\n"
                    )
                    break

                # ── Stage 3: Recommendation conversation ─────────────────
                conversation_reco = initialize_conv_reco(validated)
                # Include user profile context
                conversation_reco.append({
                    "role": "user",
                    "content": f"My requirements: {json.dumps(user_requirements)}"
                })

                reco = get_chat_completions(conversation_reco)
                conversation_reco.append({"role": "assistant", "content": reco})

                if moderation_check(reco) == 'Flagged':
                    print("Assistant: An issue occurred. Please restart.\n")
                    break

                print("\nAssistant:", reco, "\n")

            else:
                # Still in conversation — no tool call yet
                print("\nAssistant:", reply, "\n")

        # ── Stage 3: Ongoing recommendation Q&A ──────────────────────────
        else:
            conversation_reco.append({"role": "user", "content": user_input})
            reco = get_chat_completions(conversation_reco)

            if moderation_check(reco) == 'Flagged':
                print("Assistant: Sorry, this message was flagged.\n")
                continue

            conversation_reco.append({"role": "assistant", "content": reco})
            print("\nAssistant:", reco, "\n")


# ── Uncomment to run the chatbot ──────────────────────────────────────────────
dialogue_mgmt_system_v2()

Classifying 20 laptops...
Done. Saved to updated_laptop_v2.csv

Assistant: Hello! I'm here to help you find the perfect laptop. What do you plan to use the laptop for? 



You:  i am a gamer and i need a laptop 



Assistant: Great! Gaming laptops often require a good GPU. Would you prefer a laptop with low, medium, or high GPU intensity for gaming? 



You:  High GPU



Assistant: Fantastic choice! In addition to a powerful GPU, display quality is essential for a great gaming experience. Would you prefer a laptop with low, medium, or high display quality? 



...


Assistant: Thank you for providing all your preferences. Lastly, could you please specify your budget for the laptop? 



You:  unser 150000



✅ Requirements captured: {'GPU intensity': 'high', 'Display quality': 'high', 'Portability': 'low', 'Multitasking': 'high', 'Processing speed': 'high', 'Budget': 150000}

Assistant: Thank you! Let me fetch the best laptops for you...


Assistant: Based on your requirements, I recommend the following laptops:

1. Lenovo ThinkPad: Ryzen 7 processor, 16GB RAM, NVIDIA GTX graphics, 14" 2560x1440 IPS display, SSD storage, Linux OS, 6-hour battery life, Price: Rs 60,000

2. Acer Predator: Intel Core i7 processor, 16GB RAM, NVIDIA GTX graphics, 17.3" 1920x1080 IPS display, SSD storage, Windows 10 OS, 5-hour battery life, Price: Rs 80,000

If you're looking for high GPU intensity, display quality, processing speed, and multitasking capabilities within your budget, these laptops offer a good balance of performance and features. Let me know if you need more information or assistance! 



You:  exit



Assistant: Thank you for using ShopAssist! Goodbye.


## Part 8: Evaluation

Same evaluation personas as v1, but now extraction uses Function Calling.

In [15]:
SCORE_MAP_EVAL = {'low': 0, 'medium': 1, 'high': 2}

def evaluate_extraction(tagged: dict, predicted: dict) -> dict:
    """
    Compare tagged ground truth with model-extracted values.
    Returns per-key match and overall score.
    """
    results = {}
    total = 0
    for key in tagged:
        if key == 'Budget':
            continue
        t_val = SCORE_MAP_EVAL.get(tagged[key], -1)
        p_val = SCORE_MAP_EVAL.get(predicted.get(key, ''), -1)
        match = p_val >= t_val
        results[key] = {"tagged": tagged[key], "predicted": predicted.get(key, '?'), "pass": match}
        if match:
            total += 1
    results['_score'] = f"{total}/{len(results)} features matched"
    return results


# ── Helper: run a scripted conversation and extract requirements ───────────────
def run_scripted_persona(inputs: list) -> dict:
    """Simulate a user persona and extract requirements via Function Calling."""
    conv = initialize_conversation()
    # Get welcome
    chat_with_function_calling(conv)
    for text in inputs:
        conv.append({"role": "user", "content": text})
        _, tool_args = chat_with_function_calling(conv)
        if tool_args:
            return tool_args
    return {}


# ── Gamer Persona ─────────────────────────────────────────────────────────────
gamer_inputs = [
    "I prefer gaming with high graphics settings at a stationary desk.",
    "I need a high-end display for immersive visuals.",
    "I run multiple games and streaming apps simultaneously.",
    "Budget is 150,000 INR."
]
gamer_tagged = {'GPU intensity': 'high', 'Display quality': 'high',
                'Portability': 'low', 'Multitasking': 'high',
                'Processing speed': 'high', 'Budget': 150000}

print("Running Gamer persona...")
gamer_extracted = run_scripted_persona(gamer_inputs)
print("Extracted:", gamer_extracted)
print("Evaluation:", json.dumps(evaluate_extraction(gamer_tagged, gamer_extracted), indent=2))

Running Gamer persona...
Extracted: {}
Evaluation: {
  "GPU intensity": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Display quality": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Portability": {
    "tagged": "low",
    "predicted": "?",
    "pass": false
  },
  "Multitasking": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Processing speed": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "_score": "0/5 features matched"
}


In [16]:
# ── Academic Researcher Persona ───────────────────────────────────────────────
academic_inputs = [
    "I'm an academic researcher working with MATLAB, Python, and computer vision.",
    "I occasionally carry my laptop to college and conferences.",
    "I need a decent display. Budget is approximately 1,00,000 INR."
]
academic_tagged = {'GPU intensity': 'high', 'Display quality': 'medium',
                   'Portability': 'medium', 'Multitasking': 'medium',
                   'Processing speed': 'high', 'Budget': 100000}

print("Running Academic persona...")
academic_extracted = run_scripted_persona(academic_inputs)
print("Extracted:", academic_extracted)
print("Evaluation:", json.dumps(evaluate_extraction(academic_tagged, academic_extracted), indent=2))

Running Academic persona...
Extracted: {}
Evaluation: {
  "GPU intensity": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Display quality": {
    "tagged": "medium",
    "predicted": "?",
    "pass": false
  },
  "Portability": {
    "tagged": "medium",
    "predicted": "?",
    "pass": false
  },
  "Multitasking": {
    "tagged": "medium",
    "predicted": "?",
    "pass": false
  },
  "Processing speed": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "_score": "0/5 features matched"
}


In [17]:
# ── Business Executive Persona ────────────────────────────────────────────────
business_inputs = [
    "I'm a business executive. I use my laptop for Zoom calls, Excel analysis, and reports.",
    "I travel frequently so portability is essential. Graphics is not important.",
    "Maximum budget is 2,00,000 INR."
]
business_tagged = {'GPU intensity': 'low', 'Display quality': 'high',
                   'Portability': 'high', 'Multitasking': 'high',
                   'Processing speed': 'high', 'Budget': 200000}

print("Running Business Executive persona...")
business_extracted = run_scripted_persona(business_inputs)
print("Extracted:", business_extracted)
print("Evaluation:", json.dumps(evaluate_extraction(business_tagged, business_extracted), indent=2))

Running Business Executive persona...
Extracted: {}
Evaluation: {
  "GPU intensity": {
    "tagged": "low",
    "predicted": "?",
    "pass": false
  },
  "Display quality": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Portability": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Multitasking": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "Processing speed": {
    "tagged": "high",
    "predicted": "?",
    "pass": false
  },
  "_score": "0/5 features matched"
}


## Part 9: Summary of Improvements

| Metric | ShopAssist v1 | ShopAssist 2.0 |
|---|---|---|
| LLM calls to extract user profile | 3 (chat + intent_confirmation + dictionary_present) | 1 (tool call built-in) |
| LLM calls for laptop classification | 1 per laptop (JSON prompt hacks) | 1 per laptop (schema-enforced) |
| Risk of malformed output | High (regex/eval parsing) | Near-zero (JSON schema validation) |
| Lines of code for extraction logic | ~80 | ~25 |
| Removed functions | — | `intent_confirmation_layer()`, `dictionary_present()` |

### Future Improvements
1. **Parallel laptop classification** — use `asyncio` or threading to classify all laptops concurrently.
2. **Streaming responses** — enable streaming in Stage 3 for a more interactive feel.
3. **Multiple tools** — add a `refine_requirements` tool so users can change requirements mid-conversation.
4. **Persistent catalogue** — store feature vectors in a vector DB for semantic similarity search.
5. **GPT-4o upgrade** — switching to `gpt-4o` improves classification accuracy especially for nuanced descriptions.